# Variant Type

Spark 4.0 introduced a new built-in type — `VARIANT` — for storing semi-structured data (typically JSON). Unlike a struct, a variant column does not require a fixed schema up front: each row can have a different shape. Under the hood the data is stored in an efficient binary format and individual fields can be extracted on demand without re-parsing JSON strings.

In this notebook you will practice the VARIANT type on the `questions` dataset.

Tasks:
1. Read the questions data as raw text (one JSON document per line).
2. Parse each line into a VARIANT column with `parse_json`.
3. Extract typed fields from the VARIANT using `variant_get`.
4. Use `try_variant_get` to safely extract a field that may not exist.
5. Discover the merged schema across all rows with `schema_of_variant_agg`.
6. Filter and aggregate directly on the VARIANT column without materializing all fields.

## When does it make sense to read JSON as text + `parse_json` instead of `spark.read.json`?

On *this* dataset — a clean, well-structured StackOverflow dump — it does **not**. If the schema is stable and known, `spark.read.json(...).schema(<known_schema>)` is strictly better: you get strong typing, per-column predicate pushdown, and compile-time field references. The text-then-`parse_json` pattern in this notebook is used to give us a clean entry point to the VARIANT API.

In production, however, the same pattern is genuinely useful in three situations:

* **Schema drift / unknown schema.** `spark.read.json` infers one schema for the whole input. When producers add a field, drop a field, or change a type (e.g. `id` is sometimes an int, sometimes a string), it either silently widens the type, drops data, or stuffs rows into `_corrupt_record`. With VARIANT each row keeps its full structure independently — two rows in the same column can have completely different shapes and both are preserved verbatim. This is exactly the situation with event/log pipelines, third-party webhooks, or any "we keep adding fields to the payload" system.
* **Heterogeneous payloads in one column.** A logs/events table where the `payload` looks different per event type. The struct alternative is a giant union struct of mostly-null fields. VARIANT stores only what each row actually has, in a compact binary format.
* **Avoiding the schema-inference scan.** `spark.read.json` without an explicit schema reads (a sample of) the data twice — once to infer, once to load. On large directories that is expensive. Reading as text and producing a VARIANT skips that pass entirely; you pay only when you actually pull fields out with `variant_get`.

Rule of thumb: VARIANT is the right tool when you *can't* commit to a schema, not as a default replacement for typed structs.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, parse_json, variant_get, try_variant_get, schema_of_variant_agg, desc
)

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Variant Type')
    .getOrCreate()
)

In [ ]:
print(spark.version)

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

questions_input_path = os.path.join(project_path, 'data/questions-json')

### Task 1: Read the questions data as raw text

Imagine that we do not know the schema of the incoming JSON documents — for example because the source system changes the structure over time. We will read each file as plain text so that every line becomes one row with a single `value` column holding the raw JSON string.

Hint:
* use `spark.read.text(<path>)`
* the resulting DataFrame has a single column `value: string`

In [ ]:
rawDF = spark.read.text(questions_input_path)

rawDF.printSchema()

rawDF.show(n=2, truncate=80)

### Task 2: Parse each line into a VARIANT column

Hint:
* `parse_json(col)` converts a JSON string into a VARIANT value
* note the new type `variant` in the printed schema
* there is also `try_parse_json` which returns `null` for malformed input instead of failing the job
* docs for [parse_json](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.parse_json.html)

In [ ]:
variantDF = rawDF.withColumn('data', parse_json(col('value'))).drop('value')

variantDF.printSchema()

variantDF.show(n=2, truncate=80)

### Task 3: Extract typed fields with `variant_get`

`variant_get(variant, path, type)` navigates the variant using a JSON-path-style expression starting with `$` and casts the result to the requested SQL type. Extract `question_id`, `title`, `score` and `creation_date`.

Hint:
* path looks like `$.field_name` (or `$[0]` for arrays, `$.field['key']`, ...)
* the third argument is the SQL type to cast to — e.g. `'bigint'`, `'string'`, `'int'`, `'timestamp'`
* docs for [variant_get](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.variant_get.html)

In [ ]:
extractedDF = (
    variantDF
    .withColumn('question_id', variant_get('data', '$.question_id', 'bigint'))
    .withColumn('title', variant_get('data', '$.title', 'string'))
    .withColumn('score', variant_get('data', '$.score', 'int'))
    .withColumn('creation_date', variant_get('data', '$.creation_date', 'timestamp'))
)

extractedDF.select('question_id', 'title', 'score', 'creation_date').show(n=5, truncate=50)

### Task 4: Safely extract a field that may not exist

When the schema is unknown, some documents may not contain a given field. Use `try_variant_get` to get `null` back instead of raising an error.

Hint:
* try to extract a non-existent field, e.g. `$.does_not_exist`
* compare what `variant_get` and `try_variant_get` return
* docs for [try_variant_get](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.try_variant_get.html)

In [ ]:
(
    variantDF
    .withColumn('maybe_missing', try_variant_get('data', '$.does_not_exist', 'string'))
    .select('maybe_missing')
).show(n=5)

### Task 5: Discover the schema across all rows

When you receive variant data from an unknown source, `schema_of_variant_agg` aggregates over the whole column and returns the merged SQL schema as a string. It is invaluable for exploration.

Hint:
* `schema_of_variant_agg(<variant column>)` is an aggregate function
* there is also a per-row variant `schema_of_variant` if you want to see how different the rows are
* docs for [schema_of_variant_agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.schema_of_variant_agg.html)

In [ ]:
inferred_schema = (
    variantDF
    .select(schema_of_variant_agg('data').alias('schema'))
    .collect()[0]['schema']
)

print(inferred_schema)

### Task 6: Query the VARIANT column directly

There is no need to materialize every field as its own column before filtering or aggregating — `variant_get` can be used in any expression. Find the five highest-scored questions using only the VARIANT column.

Hint:
* sort by `variant_get('data', '$.score', 'int')`
* select a couple of fields with `variant_get` in the same projection
* note that you only pay the binary-decoding cost for the fields you actually ask for

In [ ]:
(
    variantDF
    .select(
        variant_get('data', '$.question_id', 'bigint').alias('question_id'),
        variant_get('data', '$.title', 'string').alias('title'),
        variant_get('data', '$.score', 'int').alias('score'),
    )
    .orderBy(desc('score'))
).show(n=5, truncate=60)

In [ ]:
spark.stop()